# Task 1 — QuadX-Hover-v4 with SAC

**Goal.** Run the final SAC Hover experiments from one notebook, while keeping the underlying training logic unchanged.

**Experimental protocol.**

1. Run the main multi-seed SAC Hover experiment used in the report.
2. Run one-seed diagnostics that vary `learning_starts` to study stability.
3. Run one-seed diagnostics that vary `buffer_size` to study replay effects.
4. Reuse the same report-style plotting and evaluation pipeline throughout.

**Note.** This notebook only reorganizes the workflow. Training itself is still delegated to `training_cell_hover_sac.py`, which now contains the SAC Hover training and evaluation pipeline directly.


## 1. Installs


In [ ]:
!pip install stable_baselines3
!pip install PyFlyt
!pip install PyBullet
!pip install tqdm
!pip install matplotlib
!pip install numpy
!pip install torch

## 2. Imports and paths


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import stable_baselines3 as sb3
import torch

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "training_cell_hover_sac.py").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    guessed = NOTEBOOK_DIR / "scripts" / "Hover"
    if guessed.exists():
        NOTEBOOK_DIR = guessed
        PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
    else:
        PROJECT_ROOT = NOTEBOOK_DIR

SCRIPTS_DIR = PROJECT_ROOT / "scripts"

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Stable-Baselines3 version: {sb3.__version__}")
print(f"PyTorch version:           {torch.__version__}")
print(f"CUDA available:            {torch.cuda.is_available()}")
print(f"Notebook dir:              {NOTEBOOK_DIR}")
print(f"Project root:              {PROJECT_ROOT}")
print(f"Scripts dir:               {SCRIPTS_DIR}")


## 3. Global settings

These are the common SAC settings shared across the Hover runs unless explicitly overridden in a diagnostic variant.


In [ ]:
# ---- Experiment identity ---------------------------------------------------
ALGO_NAME = "SAC"
ENV_NAME = "hover"
FLIGHT_MODE = 0

# ---- Compute budget --------------------------------------------------------
SEEDS = [0, 1, 2, 3, 4]
TIMESTEPS = 500_000

# ---- SAC hyperparameters ---------------------------------------------------
LEARNING_RATE = 5e-5
BUFFER_SIZE = 1_000_000
BATCH_SIZE = 512
GAMMA = 0.995
ENT_COEF = "auto"
LEARNING_STARTS = 50_000
GRADIENT_STEPS = 1

# ---- Evaluation ------------------------------------------------------------
EVAL_FREQ = 10_000
N_EVAL_EPISODES = 30
FINAL_EVAL_EPISODES = 50

# ---- Output paths ----------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results"
SAVE_NAME = None

# ---- Behaviour flags -------------------------------------------------------
FORCE_RETRAIN = False
PLOT_ONLY = False

## 4. Experiment registry

We define all Hover SAC experiments here so that the main report run and the one-seed diagnostics live in one place. The report baseline uses five seeds, while the `learning_starts` and `buffer_size` diagnostics are one-seed runs intended for interpretation rather than statistical comparison.


In [ ]:
from dataclasses import replace

from training_cell_hover_sac import HoverTrainingConfig, output_paths, train_all

BASELINE_CFG = HoverTrainingConfig(
    algo_name=ALGO_NAME,
    env_name=ENV_NAME,
    flight_mode=FLIGHT_MODE,
    timesteps=TIMESTEPS,
    learning_rate=LEARNING_RATE,
    buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    gamma=GAMMA,
    ent_coef=ENT_COEF,
    learning_starts=LEARNING_STARTS,
    gradient_steps=GRADIENT_STEPS,
    seeds=SEEDS,
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    final_eval_episodes=FINAL_EVAL_EPISODES,
    project_root=PROJECT_ROOT,
    results_dir=RESULTS_DIR,
    save_name=SAVE_NAME,
)

LS10K_CFG = replace(BASELINE_CFG, seeds=[0], learning_starts=10_000, save_name="SAC_hover_mode0_500k_ls10k")
LS100K_CFG = replace(BASELINE_CFG, seeds=[0], learning_starts=100_000, save_name="SAC_hover_mode0_500k_ls100k")
BUF100K_CFG = replace(BASELINE_CFG, seeds=[0], buffer_size=100_000, save_name="SAC_hover_mode0_500k_buf100k")
BUF300K_CFG = replace(BASELINE_CFG, seeds=[0], buffer_size=300_000, save_name="SAC_hover_mode0_500k_buf300k")

RUN_BASELINE = True
RUN_LS_DIAGNOSTICS = False
RUN_BUFFER_DIAGNOSTICS = False

EXPERIMENTS = {
    "baseline": BASELINE_CFG,
    "ls10k": LS10K_CFG,
    "ls100k": LS100K_CFG,
    "buf100k": BUF100K_CFG,
    "buf300k": BUF300K_CFG,
}

for name, cfg in EXPERIMENTS.items():
    print(name, cfg.resolved_save_name(), cfg.seeds, cfg.learning_starts, cfg.buffer_size)


## 5. Main report run

This is the main multi-seed Hover SAC experiment used in the report figures and summary statistics.


In [ ]:
baseline_outputs = {}
if RUN_BASELINE:
    baseline_outputs = train_all(BASELINE_CFG, plot_only=PLOT_ONLY, force_retrain=FORCE_RETRAIN)
else:
    print("Baseline run skipped (RUN_BASELINE=False).")
baseline_outputs


## 6. Learning-starts diagnostics

These one-seed runs test whether delaying replay warmup changes the stability of SAC on Hover.


In [ ]:
learning_starts_outputs = {}
if RUN_LS_DIAGNOSTICS:
    for name, cfg in [("ls10k", LS10K_CFG), ("ls100k", LS100K_CFG)]:
        print(f"\n===== Running {name} =====")
        learning_starts_outputs[name] = train_all(cfg, plot_only=PLOT_ONLY, force_retrain=FORCE_RETRAIN)
else:
    print("Learning-starts diagnostics skipped (RUN_LS_DIAGNOSTICS=False).")
learning_starts_outputs


## 7. Buffer-size diagnostics

These one-seed runs test how replay-buffer size affects the final Hover SAC behavior.


In [ ]:
buffer_outputs = {}
if RUN_BUFFER_DIAGNOSTICS:
    for name, cfg in [("buf100k", BUF100K_CFG), ("buf300k", BUF300K_CFG)]:
        print(f"\n===== Running {name} =====")
        buffer_outputs[name] = train_all(cfg, plot_only=PLOT_ONLY, force_retrain=FORCE_RETRAIN)
else:
    print("Buffer-size diagnostics skipped (RUN_BUFFER_DIAGNOSTICS=False).")
buffer_outputs


## 8. Inspect outputs

Use the helper below to display the main artifacts for any configured experiment. This keeps the baseline and the diagnostics in the same notebook without duplicating plotting code.


In [ ]:
import pandas as pd
from IPython.display import Image, display

def display_artifacts(cfg):
    paths = output_paths(cfg)
    for name, path in paths.items():
        print(f"{name}: {path}  exists={path.exists()}")

    if paths["learning_curve"].exists():
        display(Image(filename=str(paths["learning_curve"])))
    if paths["final_boxplot"].exists():
        display(Image(filename=str(paths["final_boxplot"])))
    if paths["final_stats"].exists():
        display(pd.read_csv(paths["final_stats"]))
    if paths["best_stats"].exists():
        display(pd.read_csv(paths["best_stats"]))
    if paths["checkpoint_comparison"].exists():
        display(pd.read_csv(paths["checkpoint_comparison"]))


In [ ]:
# Examples:
display_artifacts(BASELINE_CFG)
# display_artifacts(LS10K_CFG)
# display_artifacts(LS100K_CFG)
# display_artifacts(BUF100K_CFG)
# display_artifacts(BUF300K_CFG)
